# ICU Mortality Prediction Model — **Phase 4 (gradient boosting)**
### XGBoost, LightGBM, and CatBoost on both sets, hyperparameter tuning, SHAP, DeLong test, calibration, and decision curve

This phase starts from the base and artifacts saved in Phase 2/Phase 3:
- `modelo_A_fase2.csv` — 5,676 index admissions, one row per patient.
- `modelo_A_conjuntos_variaveis.json` — Set 1 (ready-made scores) and Set 2 (raw components).
- `modelo_A_fase3_linha_de_base.csv` — logistic regression and SAPS3 performance, for direct comparison.

Deliverables of this phase:
1. Gradient boosting (XGBoost, LightGBM, CatBoost) on both sets, with categoricals and missingness handled natively by each engine (no one-hot, no imputation).
2. Hyperparameter tuning by stratified, grouped cross-validation (`StratifiedGroupKFold`, 5 folds), on training only.
3. No imbalance correction (`scale_pos_weight` unused, in any engine) — a decision documented in the Technical Note on imbalance in gradient boosting, which extends to boosting the rationale already validated for logistic regression in Phase 3.
4. Two validations (temporal as the primary verdict; random by patient as reference), always compared to the Phase 3 baseline and to SAPS3.
5. SHAP for the best engine of each set, separately.
6. DeLong test against SAPS3.
7. Fine calibration (decomposed Brier, calibration curve, slope, and intercept).
8. Decision-curve analysis (net benefit).

**Note on the outcome:** all modeling in this phase uses `mortalidade_hospitalar` (primary definition,
`HospitalDischargeName`). Cardiovascular patients transferred before death may be coded as survivors
under this definition — a transfer bias documented in the Bias Note. This phase's results (AUROC,
DeLong, SHAP, calibration, decision curve) refer to the primary definition unless explicitly stated.
The parallel analysis with `mortalidade_hospitalar_ajustada` will be run in Phase 5 (sensitivity
notebook), following the same protocol as this phase.

In [ ]:
!pip -q install xgboost lightgbm catboost shap --upgrade

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

# === Configuration ===
FASE2_DIR  = 'outputs/phase2'
FASE3_DIR  = 'outputs/phase3'
OUTDIR     = 'outputs/phase4'
os.makedirs(OUTDIR, exist_ok=True)

CSV_PATH   = f'{FASE2_DIR}/modelo_A_fase2.csv'
CONJ_PATH  = f'{FASE2_DIR}/modelo_A_sets_variaveis.json'
BASELINE_PATH = f'{FASE3_DIR}/modelo_A_fase3_linha_de_base.csv'

ALVO = 'mortalidade_hospitalar'
CORTE_TEMPORAL = 2023   # training < 2023; test >= 2023 — same cut as Phase 3
SEED = 42

CATEGORICAS = ['Gender','AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
               'AdmissionMainDiagnosisName_grp','Chronic Health Status Name','Status COVID 19',
               'OrigemPagamento']

print('Configuration loaded.')

## 1 · Load the base and reconstruct the two validations

Same base and same splits as Phase 3 — reconstructed here so that this notebook is self-contained and
the results remain directly comparable to the baseline.

In [ ]:
df = pd.read_csv(CSV_PATH)

# Preventive sanitization — idempotent: changes nothing if the base is already clean.
# Protects against the case where the CSV was opened and re-saved by Excel between phases,
# reintroducing thousands-separator artifacts (same logic as Phase 3).
def _tem_milhar(serie):
    return serie.astype(str).str.count(r'\.').ge(2).sum()

col_ratio = 'Highest Pa O2Fi O21h'
if col_ratio in df.columns and _tem_milhar(df[col_ratio]) > 0:
    df[col_ratio] = (pd.to_numeric(df['Highest Pa O21h'], errors='coerce') /
                     (pd.to_numeric(df['Highest Fi O21h'], errors='coerce') / 100))
    print(f'⚠️  {col_ratio}: recalculated (thousands-separator artifact detected)')

for _col in df.select_dtypes('object').columns:
    if _col == col_ratio: continue
    if _tem_milhar(df[_col]) > 0:
        def _fix(v):
            if str(v).count('.') < 2: return v
            p = str(v).split('.')
            return float(p[0] + '.' + ''.join(p[1:]))
        df[_col] = df[_col].map(_fix)
        print(f'⚠️  {_col}: thousands-separator artifact fixed')

print('Sanitization complete (no message above = clean base).')

# Rebuild adm_datetime — consistency with previous phases
if 'adm_datetime' in df.columns:
    df['adm_datetime'] = pd.to_datetime(df['adm_datetime'], errors='coerce')

with open(CONJ_PATH, encoding='utf-8') as f:
    sets = json.load(f)

CONJUNTO_1 = sets['set_1_escores']
CONJUNTO_2 = sets['set_2_bruto']
BENCHMARK  = sets['benchmark']   # 'Saps3Death Probability Standard Equation'

# year column, from 'Month Year' (same logic as Phase 3)
if 'ano' not in df.columns:
    df['ano'] = pd.to_datetime(df['Month Year']).dt.year

# grouping identifier for cross-validation — in this cohort (index admission)
# each patient contributes a single row; StratifiedGroupKFold uses the row index itself as group
# when 'patient_id' is not present, which is logically equivalent to StratifiedKFold here,
# but keeps the code ready for reuse in sensitivity analyses with multiple admissions.
GROUP_COL = 'patient_id' if 'patient_id' in df.columns else None
if GROUP_COL is None:
    df['patient_id'] = df.index
    GROUP_COL = 'patient_id'

print(f'Base carregada: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Set 1: {len(CONJUNTO_1)} variables  |  Set 2: {len(CONJUNTO_2)} variables')

In [ ]:
# --- Temporal validation (primary) ---
treino_t = df[df['ano'] <  CORTE_TEMPORAL].copy()
teste_t  = df[df['ano'] >= CORTE_TEMPORAL].copy()

# --- Random validation by patient (reference) — same random_state as Phase 3 ---
# Index-admission cohort: one row per patient — a row-level train_test_split
# equals a patient-level split, with no between-group leakage.
treino_a, teste_a = train_test_split(df, test_size=0.25, random_state=SEED, stratify=df[ALVO])

for nome, tr, te in [('temporal', treino_t, teste_t), ('random', treino_a, teste_a)]:
    tot = len(tr) + len(te)
    print(f'VALIDATION {nome.upper()} — training: {len(tr)} ({len(tr)/tot*100:.1f}%) | '
          f'test: {len(te)} ({len(te)/tot*100:.1f}%) | training deaths: {tr[ALVO].mean()*100:.1f}% | '
          f'test deaths: {te[ALVO].mean()*100:.1f}%')

## 1.1 · Group the main diagnosis

`modelo_A_fase2.csv` stores the raw `AdmissionMainDiagnosisName` (~503 text categories) — the grouping
into the 20 most frequent categories + "other" was done in Phase 3 only in memory, inside that notebook,
and never written back to the CSV. `CATEGORICAS` already expects the grouped version
(`AdmissionMainDiagnosisName_grp`), so the grouping must be redone here, **separately within each
train/test pair** (temporal and random) — the most frequent categories learned on training do not leak
into test, and are not necessarily the same across the two validations.

In [ ]:
N_TOP_DIAG = 20

def agrupar_diagnostico_principal(treino, teste, col='AdmissionMainDiagnosisName'):
    col_grp = f'{col}_grp'
    # top categories learned on training only — no leakage
    top_categorias = treino[col].value_counts().nlargest(N_TOP_DIAG).index
    treino = treino.copy(); teste = teste.copy()
    treino[col_grp] = treino[col].where(treino[col].isin(top_categorias), 'outros')
    teste[col_grp]  = teste[col].where(teste[col].isin(top_categorias), 'outros')
    return treino, teste

treino_t, teste_t = agrupar_diagnostico_principal(treino_t, teste_t)
treino_a, teste_a = agrupar_diagnostico_principal(treino_a, teste_a)

# SET_1/2 must point to the grouped version, not the raw one
def usar_diagnostico_agrupado(set):
    return ['AdmissionMainDiagnosisName_grp' if c == 'AdmissionMainDiagnosisName' else c for c in set]

CONJUNTO_1 = usar_diagnostico_agrupado(CONJUNTO_1)
CONJUNTO_2 = usar_diagnostico_agrupado(CONJUNTO_2)

print('Main diagnosis grouped (top 20 + "other") in the four training/test bases.')

## 2 · Data preparation for boosting

No one-hot, no imputation, no scaling — the three engines handle categoricals and missingness natively,
which makes them more faithful to the original structure of clinical data than Phase 3's logistic
regression. Categoricals are converted to the `category` dtype; numeric missing values remain `NaN`
(the missingness indicators built in Phase 2 preserve the information of where data was absent). For
CatBoost, which requires categories without `NaN`, blanks in categorical columns receive the explicit
label `"Não informado"` (Not reported).

In [ ]:
def preparar_boosting(set, treino, teste):
    cats = [c for c in set if c in CATEGORICAS]
    nums = [c for c in set if c not in CATEGORICAS and c != 'Month Year']
    cols = cats + nums

    Xtr = treino[cols].copy()
    Xte = teste[cols].copy()

    for c in cats:
        Xtr[c] = Xtr[c].astype('object').fillna('Not reported')
        Xte[c] = Xte[c].astype('object').fillna('Not reported')

        # categories seen in training + the missingness label, always present
        categorias = pd.Index(Xtr[c].unique()).union(['Not reported'])

        # any test category not seen in training also becomes "Not reported" (Not reported)
        # (instead of NaN) — avoids a new category with no learned risk
        Xte[c] = Xte[c].where(Xte[c].isin(categorias), 'Not reported')

        Xtr[c] = pd.Categorical(Xtr[c], categories=categorias)
        Xte[c] = pd.Categorical(Xte[c], categories=categorias)

    return Xtr, Xte, cats, nums

print('Preparation function (native, no one-hot/imputation) defined.')

## 3 · Engines and hyperparameter grids

Moderate grids, compatible with the cohort size (~4,300 training rows). `scale_pos_weight` /
`class_weights` do not enter any grid — a decision documented in the Technical Note on imbalance.

In [ ]:
GRADES = {
    'xgboost': {
        'n_estimators': [200, 300, 400, 600, 800],
        'max_depth': [2, 3, 4, 5, 6],
        'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5, 10, 20],
        'reg_lambda': [0.5, 1, 2, 5, 10],
        'reg_alpha': [0, 0.1, 0.5, 1],
    },
    'lightgbm': {
        'n_estimators': [200, 300, 400, 600, 800],
        'max_depth': [-1, 3, 4, 5, 6],
        'num_leaves': [15, 31, 63, 127],
        'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_samples': [5, 10, 20, 30, 50],
        'reg_lambda': [0, 0.5, 1, 2, 5],
    },
    'catboost': {
        'iterations': [200, 300, 400, 600, 800],
        'depth': [3, 4, 5, 6, 7],
        'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
        'l2_leaf_reg': [1, 3, 5, 7, 9],
        'border_count': [32, 64, 128],
    },
}

N_ITER_BUSCA = 30
print('Hyperparameter grids defined.')

## 4 · Hyperparameter tuning (`StratifiedGroupKFold`, 5 folds, training only)

One function per engine, all following the same contract: they receive training `X`, `y`, `groups`, and
return the best estimator (refit over the full training set) and the winning hyperparameters.

**On parallelism**: only `RandomizedSearchCV` uses all cores (`n_jobs=-1`); each engine, inside it, runs
on a single thread (`n_jobs=1` / `thread_count=1`). This avoids nested parallelism — two levels of
processes competing for the same cores at the same time — which can make the search extremely slow and,
in LightGBM's specific case, deadlock entirely (a known library deadlock when combined with
scikit-learn's parallelism).

In [ ]:
def tunar_xgboost(X, y, groups, cats):
    base = xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', enable_categorical=True,
        scale_pos_weight=1,  # no imbalance correction (see Technical Note)
        random_state=SEED, n_jobs=1,  # parallelism stays in RandomizedSearchCV only, not nested
    )
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    busca = RandomizedSearchCV(base, GRADES['xgboost'], n_iter=N_ITER_BUSCA, scoring='roc_auc',
                                cv=cv, random_state=SEED, n_jobs=-1, refit=True)
    busca.fit(X, y, groups=groups)
    return busca.best_estimator_, busca.best_params_, busca.best_score_


def tunar_lightgbm(X, y, groups, cats):
    base = lgb.LGBMClassifier(
        objective='binary', random_state=SEED, n_jobs=1, verbosity=-1,
        class_weight=None,  # no imbalance correction
        # n_jobs=1 here is essential: LightGBM has a known deadlock when its
        # own internal parallelism adds to RandomizedSearchCV's parallelism
    )
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    busca = RandomizedSearchCV(base, GRADES['lightgbm'], n_iter=N_ITER_BUSCA, scoring='roc_auc',
                                cv=cv, random_state=SEED, n_jobs=-1, refit=True)
    busca.fit(X, y, groups=groups, categorical_feature=cats)
    return busca.best_estimator_, busca.best_params_, busca.best_score_


def tunar_catboost(X, y, groups, cats):
    base = CatBoostClassifier(
        loss_function='Logloss', random_seed=SEED, verbose=False,
        thread_count=1,  # parallelism stays in RandomizedSearchCV only, not nested
        # cat_features does NOT go in the constructor — CatBoost's get_params() does not
        # return it in a way sklearn.clone() recognizes as identical, and
        # RandomizedSearchCV fails to clone the estimator at each fold. The parameter
        # goes in fit(), which is where CatBoost actually expects to receive it.
        # (no class_weights — see Technical Note on imbalance)
    )
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    busca = RandomizedSearchCV(base, GRADES['catboost'], n_iter=N_ITER_BUSCA, scoring='roc_auc',
                                cv=cv, random_state=SEED, n_jobs=-1, refit=True)
    busca.fit(X, y, groups=groups, cat_features=cats)
    return busca.best_estimator_, busca.best_params_, busca.best_score_


TUNADORES = {'XGBoost': tunar_xgboost, 'LightGBM': tunar_lightgbm, 'CatBoost': tunar_catboost}
print('Tuning functions defined for the three engines.')

## 5 · Train and evaluate — three engines, two sets, two validations

A single function sweeps engine × set × validation, tuning hyperparameters on the corresponding training
split and evaluating on the corresponding test split.

**Automatic checkpointing**: each engine × set × validation combination is saved to disk
(`outputs/phase4/checkpoints/`) as soon as it finishes training. If the Colab session dies mid-run, simply
rerun this cell — completed combinations are loaded from the checkpoint instead of retrained, and only
what is missing is computed. To force a full retrain, clear `CHECKPOINT_DIR` before running.

In [ ]:
import re, joblib

CHECKPOINT_DIR = f'{OUTDIR}/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _slug(*parts):
    s = '_'.join(parts)
    return re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_')

def _caminho_checkpoint(nome_validacao, nome_set, nome_motor):
    base = _slug(nome_validacao, nome_set, nome_motor)
    return f'{CHECKPOINT_DIR}/{base}_modelo.joblib', f'{CHECKPOINT_DIR}/{base}_info.json'


def rodar_set_validacao(set, nome_set, treino, teste, nome_validacao):
    Xtr, Xte, cats, nums = preparar_boosting(set, treino, teste)
    ytr, yte = treino[ALVO].values, teste[ALVO].values
    groups_tr = treino[GROUP_COL].values

    linhas, modelos, predicoes = [], {}, {}
    for nome_motor, tunar in TUNADORES.items():
        caminho_modelo, caminho_info = _caminho_checkpoint(nome_validacao, nome_set, nome_motor)

        if os.path.exists(caminho_modelo) and os.path.exists(caminho_info):
            modelo = joblib.load(caminho_modelo)
            with open(caminho_info, encoding='utf-8') as f:
                linha = json.load(f)['linha']
            tag = '(checkpoint)'
        else:
            modelo, params, cv_auroc = tunar(Xtr, ytr, groups_tr, cats)
            p_treino_ckpt = modelo.predict_proba(Xte)[:, 1]
            linha = {
                'validacao': nome_validacao, 'set': nome_set, 'motor': nome_motor,
                'AUROC': roc_auc_score(yte, p_treino_ckpt), 'AUPRC': average_precision_score(yte, p_treino_ckpt),
                'Brier': brier_score_loss(yte, p_treino_ckpt), 'AUROC_cv_treino': cv_auroc,
                'n_variaveis': Xtr.shape[1],
            }
            joblib.dump(modelo, caminho_modelo)
            with open(caminho_info, 'w', encoding='utf-8') as f:
                json.dump({'linha': linha, 'params': params}, f, ensure_ascii=False, indent=2)
            tag = ''

        p = modelo.predict_proba(Xte)[:, 1]
        linhas.append(linha)
        modelos[(nome_set, nome_motor, nome_validacao)] = {'modelo': modelo, 'X_treino': Xtr,
                                                                  'X_teste': Xte, 'cats': cats}
        predicoes[(nome_set, nome_motor, nome_validacao)] = (p, yte)
        print(f"  {nome_validacao:10s} | {nome_set:20s} | {nome_motor:9s} | {tag} "
              f"AUROC={linha['AUROC']:.3f}  AUPRC={linha['AUPRC']:.3f}  Brier={linha['Brier']:.3f}")
    return linhas, modelos, predicoes


resultados_gb, modelos_gb, predicoes_gb = [], {}, {}
for nome_validacao, treino, teste in [('temporal', treino_t, teste_t), ('aleatoria_por_paciente', treino_a, teste_a)]:
    for conj, nome_set in [(CONJUNTO_1, 'Conjunto 1 (escores)'), (CONJUNTO_2, 'Conjunto 2 (brutos)')]:
        linhas, modelos, predicoes = rodar_set_validacao(conj, nome_set, treino, teste, nome_validacao)
        resultados_gb.extend(linhas)
        modelos_gb.update(modelos)
        predicoes_gb.update(predicoes)

tab_gb = pd.DataFrame(resultados_gb)
print()
print(tab_gb.to_string(index=False))

## 6 · Comparison with the baseline (Phase 3) and SAPS3

In [ ]:
baseline = pd.read_csv(BASELINE_PATH)
baseline = baseline.rename(columns={'nome': 'set'})
baseline['motor'] = baseline['set'].apply(
    lambda x: 'Logistic regression' if 'SAPS3' not in x else 'SAPS3')
baseline['set'] = baseline['set'].str.replace(' (benchmark)', '', regex=False)

comparacao = pd.concat([tab_gb, baseline], ignore_index=True)
comparacao = comparacao.sort_values(['validacao', 'set', 'AUROC'], ascending=[True, True, False])
comp_path = f'{OUTDIR}/modelo_A_fase4_comparacao_completa.csv'
comparacao.to_csv(comp_path, index=False, encoding='utf-8-sig')
print(f'Comparison table saved: {comp_path}')
print()
print(comparacao.to_string(index=False))

### Notes on the comparison table

**`AUROC_cv_treino = NaN` for logistic regression and SAPS3**

The `AUROC_cv_treino` column records the mean AUROC obtained during hyperparameter-tuning cross-validation
(`RandomizedSearchCV`). This value exists only for the three gradient boosting engines, which undergo
hyperparameter search. Logistic regression uses fixed parameters (`penalty='l2'`, `C=1.0`) and SAPS3 is
not trained — hence the field is blank for both.

**`n_variaveis = NaN` for SAPS3**

SAPS3 uses `Saps3Death Probability Standard Equation` directly as the probability, without passing
through any variable-preparation function. There is no predictor set to count.

**Difference in variable counts between gradient boosting (132) and logistic regression (107–109)**

The three boosting engines receive categoricals as native `category` dtype — no expansion. Each
categorical column remains a single column, regardless of the number of categories. The total of 132 is
the raw column count of Set 2.

Logistic regression (Phase 3) applies `pd.get_dummies()` with `drop_first=True` — each categorical with
*k* categories becomes *k − 1* binary columns. The net result (107–109) is smaller than 132 because
Set 2's categoricals have, on average, few unique categories: the one-hot expansion creates fewer columns
than originally existed, and `drop_first` still removes one per variable.

The 2-column difference between the two validations (107 in temporal, 109 in random) reflects the fact
that one-hot is learned **on the training split only** of each validation. Since the two training sets
have different compositions — the temporal goes up to 2022, the random draws 75% of the base — rare
categories may be present in one training set and absent in the other, creating or eliminating extra
columns. With `AdmissionMainDiagnosisName_grp` already consolidated into top-20 + "other", the residual
variations likely come from `Status COVID 19` or `Chronic Health Status Name`.

These differences do not represent errors — they are the expected consequence of two distinct preparation
methods applied to the same variable set: gradient boosting does not need one-hot encoding; logistic
regression does, being a linear model that does not natively handle categoricals.

## 7 · Best-engine selection per set (temporal validation)

Temporal validation is the primary verdict (Phase 3, Section 4); the selection of the best engine for
SHAP, DeLong, fine calibration, and the decision curve follows this criterion.

In [ ]:
melhor_por_set = {}
for nome_set in ['Conjunto 1 (escores)', 'Conjunto 2 (brutos)']:
    sub = tab_gb[(tab_gb['validacao'] == 'temporal') & (tab_gb['set'] == nome_set)]
    linha_melhor = sub.loc[sub['AUROC'].idxmax()]
    melhor_por_set[nome_set] = linha_melhor['motor']
    print(f"{nome_set}: melhor motor = {linha_melhor['motor']} "
          f"(AUROC={linha_melhor['AUROC']:.3f} in the temporal validation)")

## 8 · DeLong test against SAPS3

Compares, in the temporal validation, the AUROC of each set's best engine against SAPS3's AUROC, based
on DeLong's method for correlated ROC curves (same test patients in both models).

In [ ]:
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_transposed, m):
    # preds_sorted_transposed: shape (k models, n samples), positives first (0:m), negatives after (m:)
    k = preds_sorted_transposed.shape[0]
    n = preds_sorted_transposed.shape[1] - m
    tx = np.empty([k, m])
    ty = np.empty([k, n])
    tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(preds_sorted_transposed[r, :m])
        ty[r, :] = _compute_midrank(preds_sorted_transposed[r, m:])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def teste_delong(y_true, p_modelo, p_saps3):
    order = np.argsort(-y_true)
    y_ord = y_true[order]
    m = int(y_ord.sum())  # positivos
    preds = np.vstack([p_modelo[order], p_saps3[order]])
    aucs, cov = _fast_delong(preds, m)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    from scipy import stats as sstats
    z = diff / np.sqrt(var) if var > 0 else np.nan
    p_valor = 2 * (1 - sstats.norm.cdf(abs(z))) if var > 0 else np.nan
    return {'AUROC_modelo': aucs[0], 'AUROC_SAPS3': aucs[1], 'diferenca': diff, 'z': z, 'p_valor': p_valor}


resultados_delong = []
for nome_set, nome_motor in melhor_por_set.items():
    p_modelo, y_teste = predicoes_gb[(nome_set, nome_motor, 'temporal')]
    p_saps3 = (teste_t[BENCHMARK] / 100).values
    r = teste_delong(teste_t[ALVO].values, p_modelo, p_saps3)
    r['set'] = nome_set
    r['motor'] = nome_motor
    resultados_delong.append(r)

tab_delong = pd.DataFrame(resultados_delong)
delong_path = f'{OUTDIR}/modelo_A_fase4_delong.csv'
tab_delong.to_csv(delong_path, index=False, encoding='utf-8-sig')
print(tab_delong.to_string(index=False))

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':10,'axes.edgecolor':'#8A897F',
    'axes.linewidth':0.8,'axes.spines.top':False,'axes.spines.right':False,
    'figure.dpi':150,'figure.facecolor':'white'
})
NEU='#6E6D66'; TEAL='#3E8E7E'

fig, ax = plt.subplots(figsize=(7.5, 5.2))

sets  = [r['set'] for r in resultados_delong]
auroc_mod  = [r['AUROC_modelo'] for r in resultados_delong]
auroc_saps = [r['AUROC_SAPS3']  for r in resultados_delong]
motores    = [r['motor']        for r in resultados_delong]
p_valores  = [r['p_valor']      for r in resultados_delong]
n_pos  = int(teste_t[ALVO].sum())
n_neg  = len(teste_t) - n_pos

def ic95(auroc):
    q1 = auroc / (2 - auroc)
    q2 = 2 * auroc**2 / (1 + auroc)
    se = np.sqrt((auroc*(1-auroc) + (n_pos-1)*(q1-auroc**2) +
                  (n_neg-1)*(q2-auroc**2)) / (n_pos*n_neg))
    return 1.96 * se

x = np.arange(len(sets)); w = 0.30

bars_mod  = ax.bar(x - w/2, auroc_mod,  w, color=TEAL, label='Melhor modelo GB',
                   edgecolor='white', linewidth=0.5)
bars_saps = ax.bar(x + w/2, auroc_saps, w, color=NEU,  label='SAPS3 (benchmark)',
                   edgecolor='white', linewidth=0.5)

for xi, au_m, au_s in zip(x, auroc_mod, auroc_saps):
    e_m = ic95(au_m); e_s = ic95(au_s)
    ax.errorbar(xi-w/2, au_m, yerr=e_m, fmt='none', color='#2C2C2A', capsize=4, lw=1.2)
    ax.errorbar(xi+w/2, au_s, yerr=e_s, fmt='none', color='#2C2C2A', capsize=4, lw=1.2)
    ax.text(xi-w/2, au_m+e_m+0.003, f'{au_m:.3f}',
            ha='center', va='bottom', fontsize=8.5)
    ax.text(xi+w/2, au_s+e_s+0.003, f'{au_s:.3f}',
            ha='center', va='bottom', fontsize=8.5)

# p-value — positioned below the top of the figure to avoid overlapping the title
for xi, p, au_m, au_s in zip(x, p_valores, auroc_mod, auroc_saps):
    e_top = ic95(max(au_m, au_s))
    y_bracket = max(au_m, au_s) + e_top + 0.018
    p_lbl = f'p = {p:.3f}' if p >= 0.001 else 'p < 0,001'
    ax.annotate('', xy=(xi-w/2, y_bracket), xytext=(xi+w/2, y_bracket),
                arrowprops=dict(arrowstyle='-', color='#8A897F', lw=0.9))
    ax.text(xi, y_bracket + 0.003, p_lbl,
            ha='center', va='bottom', fontsize=8.5, color='#5F5E5A')

ax.set_xticks(x)
ax.set_xticklabels(
    [f'{c}\n({m})' for c, m in zip(sets, motores)],
    fontsize=9)
ax.set_ylabel('AUROC (IC 95%)')
ax.set_ylim(0.78, 0.98)   # extra headroom so the p-value does not touch the title
ax.set_title('DeLong test — best model vs. SAPS3\n(temporal validation)',
             fontsize=11, fontweight='bold', loc='left', pad=10)

# legend below the plot, outside the plotting area
ax.legend(frameon=False, fontsize=9,
          loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2)

fig.subplots_adjust(bottom=0.22)

fig_delong_path = f'{OUTDIR}/fase4_delong_comparacao.png'
plt.savefig(fig_delong_path, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Figura salva: {fig_delong_path}')

## 9 · Fine calibration

Three elements, for each set's best engine, in the temporal validation:
- **Calibration curve** (predicted probability vs. observed frequency, 10 quantile bins).
- **Calibration slope and intercept** — logistic regression of `y` on `logit(p)`: slope 1 and intercept 0
  indicate perfect calibration.
- **Brier score decomposition** (Murphy): reliability, resolution, and uncertainty.

In [ ]:
def inclinacao_intercepto_calibracao(y, p, eps=1e-6):
    p_clip = np.clip(p, eps, 1 - eps)
    logit_p = np.log(p_clip / (1 - p_clip)).reshape(-1, 1)
    modelo_cal = LogisticRegression(penalty=None, solver='lbfgs')
    modelo_cal.fit(logit_p, y)
    return modelo_cal.intercept_[0], modelo_cal.coef_[0][0]


def brier_decomposto(y, p, n_bins=10):
    frac_pos, mean_pred, contagens = [], [], []
    bins = np.quantile(p, np.linspace(0, 1, n_bins + 1))
    bins[-1] += 1e-9
    idx = np.digitize(p, bins[1:-1])
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        frac_pos.append(y[mask].mean())
        mean_pred.append(p[mask].mean())
        contagens.append(mask.sum())
    frac_pos, mean_pred, contagens = map(np.array, (frac_pos, mean_pred, contagens))
    n = contagens.sum()
    ybar = y.mean()
    reliability = np.sum(contagens * (mean_pred - frac_pos) ** 2) / n
    resolution  = np.sum(contagens * (frac_pos - ybar) ** 2) / n
    uncertainty = ybar * (1 - ybar)
    return {'reliability': reliability, 'resolution': resolution, 'uncertainty': uncertainty,
            'brier_reconstruido': reliability - resolution + uncertainty}


linhas_calibracao = []
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for eixo, (nome_set, nome_motor) in zip(axes, melhor_por_set.items()):
    p, y = predicoes_gb[(nome_set, nome_motor, 'temporal')]
    intercepto, inclinacao = inclinacao_intercepto_calibracao(y, p)
    decomp = brier_decomposto(y, p)
    linhas_calibracao.append({'set': nome_set, 'motor': nome_motor,
                               'intercepto_calibracao': intercepto, 'inclinacao_calibracao': inclinacao,
                               **decomp})
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy='quantile')
    eixo.plot(mean_pred, frac_pos, 'o-', label=f'{nome_motor}')
    eixo.plot([0, 1], [0, 1], '--', color='gray')
    eixo.set_xlabel('Probabilidade prevista')
    eixo.set_ylabel('Observed frequency')
    eixo.set_title(f'{nome_set}\nslope={inclinacao:.2f}  intercept={intercepto:.2f}')
    eixo.set_xlim(0, 1); eixo.set_ylim(0, 1)
plt.tight_layout()
fig_calibracao_path = f'{OUTDIR}/fase4_calibracao_fina.png'
fig.savefig(fig_calibracao_path, dpi=150, bbox_inches='tight')
plt.show()

tab_calibracao = pd.DataFrame(linhas_calibracao)
calibracao_path = f'{OUTDIR}/modelo_A_fase4_calibracao.csv'
tab_calibracao.to_csv(calibracao_path, index=False, encoding='utf-8-sig')
print(tab_calibracao.to_string(index=False))

### Interpretation of the calibration results

**Set 1 (ready-made scores) — LightGBM**

The slope of **1.00** indicates the model ranks risks at the right pace —
sicker patients receive proportionally higher probabilities than less sick ones, without scale distortion.
The intercept of **−0.14** reveals a systematic upward shift: the model slightly **overestimates**
absolute risk on average. Among the 10% of patients with highest predicted risk, the model predicts a
mean probability of 53.7%, against an observed real mortality of 48.6% in that group — an excess of
5.2 percentage points. This pattern is expected: the model was trained on a cohort with ~15.6% mortality
(training up to 2022) and evaluated on a cohort with ~12.2% mortality (2023+) — the prevalence drop
between training and validation pushes predicted probabilities above the real values.

**Set 2 (raw components) — XGBoost**

The slope of **0.97** is near ideal, preserving good ranking ability.
The intercept of **+0.27** indicates the opposite pattern to Set 1: the model **underestimates** risk on
average, more pronouncedly. Among the 10% of patients with highest predicted risk, the model predicts a
mean probability of 62.4%, against an observed real mortality of 69.3% in that group — a deficit of
6.9 percentage points, larger than Set 1's equivalent deviation. This effect is clinically relevant
because it occurs precisely in the sickest patient range, where calibration matters most for medical
decision-making; it possibly reflects that Set 2 — with more acute-physiology variables — captured the
extreme severity observed in the test period less completely, relative to what it had learned in the
training era.

**Implication for Phase 5**

The protocol defines the recalibration trigger as the bootstrap 95% CI of the intercept excluding zero.
Both sets satisfy this condition (intercepts of −0.14 and +0.27), indicating that post-hoc recalibration
by Platt scaling is indicated for both. Recalibration will adjust absolute probabilities toward the
diagonal without changing the AUROC — the ranking of patients by risk does not change; only the absolute
probabilities are corrected to reflect the current prevalence of the validation set.

**Brier score decomposition (Murphy)**

The total Brier Score decomposes into three independent components:
- **Reliability** (calibration loss): the error attributable to the deviation between predicted and
  observed probabilities — what recalibration will reduce.
- **Resolution**: how well the model separates risk groups — tied to discrimination and unaffected by
  recalibration.
- **Uncertainty**: intrinsic variability of the outcome (a function of the 12.2% test prevalence) —
  constant for any model in this cohort, not reducible by any intervention.

The relation `Brier = Reliability − Resolution + Uncertainty` shows that a model improves its Brier by
increasing resolution (better discrimination) or reducing reliability loss (better calibration) — the
two levers that Phase 4 and Phase 5 work on, respectively.

## 10 · SHAP — best engine of each set, separately

`TreeExplainer` works uniformly for the three engines. Summary plots (global importance) are generated
one per set, on that set's winning model in the temporal validation.

In [ ]:
import shap

shap_values_por_set = {}
for nome_set, nome_motor in melhor_por_set.items():
    info   = modelos_gb[(nome_set, nome_motor, 'temporal')]
    modelo = info['modelo']
    X_teste = info['X_teste']

    explainer   = shap.TreeExplainer(modelo)
    valores_shap = explainer.shap_values(X_teste)
    shap_values_por_set[nome_set] = (valores_shap, X_teste)

    # separate figure per set — not mixed with calibration
    fig, ax = plt.subplots(figsize=(9, 7))
    shap.summary_plot(
        valores_shap, X_teste,
        show=False,
        max_display=20,
        plot_size=None   # uses the already-created fig/ax
    )
    ax.set_xlabel('Valor SHAP (impacto na probabilidade prevista)', fontsize=10)
    ax.set_title(
        f'Variable importance — SHAP\n{nome_set} · {nome_motor} · temporal validation',
        fontsize=11, fontweight='bold', loc='left', pad=12
    )
    plt.tight_layout()
    slug = nome_set.split()[1].strip('()')
    fig_path = f'{OUTDIR}/fase4_shap_{slug}.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    print(f'Figura salva: {fig_path}')

# Transfer-bias verification via SHAP
print('\nSHAP importance verification — cardiovascular subgroup:')
for nome_c, (vals_shap, X_te) in shap_values_por_set.items():
    for col_sus in ['comorb_cardiovascular', 'AdmissionReasonName']:
        cols_list = list(X_te.columns)
        idxs = [j for j, col in enumerate(cols_list)
                if col == col_sus or col.startswith(col_sus + '_')]
        if idxs:
            imp = np.abs(vals_shap[:, idxs]).mean()
            print(f'  {nome_c} | {col_sus}: mean SHAP importance = {imp:.4f}')
        else:
            print(f'  {nome_c} | {col_sus}: column not found in the set')

### Finding — SHAP importance of comorb_cardiovascular

`comorb_cardiovascular` shows a mean SHAP importance of 0.0031 in Set 2 —
an anomalously low value for a variable with established clinical relevance in the ICU.

Two competing hypotheses:

1. **Transfer bias:** cardiovascular patients transferred before death are coded as survivors under the
   primary definition, diluting this variable's predictive signal. Testable hypothesis in Phase 5: if
   importance increases with `mortalidade_hospitalar_ajustada`, the bias is confirmed as the cause.

2. **Physiological redundancy:** Set 2's acute-severity variables (blood pressure, lactate, shock index,
   Glasgow) may capture cardiovascular severity more directly than the comorbidity flag, making it
   redundant in the model.

**Action in Phase 5:** compare the SHAP importance of `comorb_cardiovascular` between the primary and
adjusted definitions. A substantial change confirms hypothesis 1; no change points to hypothesis 2.

In [ ]:
# Distribution of comorb_cardiovascular by outcome — primary definition
print(pd.crosstab(
    df['comorb_cardiovascular'],
    df['mortalidade_hospitalar'],
    normalize='index'
).round(3))

**Prior check in the data (primary definition):**
Mortality of 13.0% without cardiovascular comorbidity vs. 15.5% with — a 2.5 pp difference, in the right
direction but smaller than clinically expected. This is consistent with physiological redundancy
(acute-severity variables capture the cardiovascular signal) possibly amplified by outcome compression
via transfer bias.
Phase 5 will test which of the two hypotheses predominates.

## 11 · Decision-curve analysis (net benefit)

Compares, at each decision threshold, the net benefit of each set's best model against SAPS3 and the two
reference strategies (treat all, treat none).

In [ ]:
def beneficio_liquido(y, p, limiares):
    n = len(y)
    beneficios = []
    for t in limiares:
        previstos_positivos = p >= t
        tp = np.sum(previstos_positivos & (y == 1))
        fp = np.sum(previstos_positivos & (y == 0))
        beneficios.append(tp / n - fp / n * (t / (1 - t)))
    return np.array(beneficios)


limiares = np.linspace(0.01, 0.5, 50)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for eixo, (nome_set, nome_motor) in zip(axes, melhor_por_set.items()):
    p_modelo, y = predicoes_gb[(nome_set, nome_motor, 'temporal')]
    p_saps3 = (teste_t[BENCHMARK] / 100).values

    bn_modelo = beneficio_liquido(y, p_modelo, limiares)
    bn_saps3  = beneficio_liquido(y, p_saps3, limiares)
    bn_todos  = y.mean() - (1 - y.mean()) * (limiares / (1 - limiares))
    bn_ninguem = np.zeros_like(limiares)

    eixo.plot(limiares, bn_modelo, label=f'{nome_motor} ({nome_set})', linewidth=2)
    eixo.plot(limiares, bn_saps3, label='SAPS3', linewidth=2)
    eixo.plot(limiares, bn_todos, '--', color='gray', label='Treat all')
    eixo.plot(limiares, bn_ninguem, ':', color='gray', label='Treat none')
    eixo.set_xlabel('Decision threshold')
    eixo.set_ylabel('Net benefit')
    eixo.set_title(nome_set)
    eixo.legend(fontsize=8)
    eixo.set_ylim(bottom=-0.05)
plt.tight_layout()
fig_dca_path = f'{OUTDIR}/fase4_curva_decisao.png'
fig.savefig(fig_dca_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_dca_path}')

### Interpretation of the decision curve (net benefit)

#### What the curve measures

The decision curve answers the central question from the clinical point of view: for each risk threshold
a physician chooses to intervene, does the model produce more benefit (correctly identified deaths) than
harm (unnecessary interventions in those who would survive)?

**Net benefit** at each threshold is computed as:

`NB = (TP/n) − (FP/n) × threshold / (1 − threshold)`

where TP are true positives, FP false positives, and n the total number of patients.
The second term is the penalty for treating those who did not need it — it grows with the threshold,
reflecting that the higher the chosen threshold, the graver the error of treating unnecessarily.

#### The four references

- **Treat none** (dotted at zero): net benefit = 0 at any threshold. It is the floor — any useful model
  must stay above this line.
- **Treat all** (gray dashed): starts at the event prevalence (~12.2%, the test-set mortality) because at
  threshold zero there is no false-positive penalty. Falls quickly as the threshold rises and the penalty
  for unnecessarily treated patients grows. Crosses zero at ~0.15 — above that threshold, treating all
  indiscriminately does more harm than good.
- **SAPS3** (orange): the current clinical benchmark.
- **Best GB model** (blue): LightGBM on Set 1, XGBoost on Set 2.

#### Reading the results

**Set 1 (ready-made scores — LightGBM):** model and SAPS3 walk together up to ~0.15.
In the 0.15–0.35 range — which corresponds to the most clinically relevant ICU thresholds — LightGBM
produces net benefit consistently superior to SAPS3. Above ~0.35, the model starts falling toward SAPS3,
a reflection of the risk overestimation in the tail identified in the calibration curve (slope 1.00,
intercept −0.14).

**Set 2 (raw components — XGBoost):** the advantage over SAPS3 is broader and more persistent. Across
practically the whole 0.05–0.50 range, the blue curve stays consistently above the orange one, at a
greater distance than in Set 1. This translates the superior AUROC (0.907 vs. 0.834) into concrete
clinical benefit: for any threshold an intensivist chooses in this range, XGBoost identifies more
patients who would truly benefit from intervention and generates fewer unnecessary interventions than
SAPS3.

Both models stay above "treat none" across the entire analyzed range —
confirming that using either of them is always superior to using no model.

#### Why the curves start near 0.122

At threshold zero, all models converge to the event prevalence (~12.2%), equal to "treat all" — there is
no false-positive penalty when the threshold is zero. Since the plot starts at threshold = 0.01, the
curves already show a minimal advantage of the models over "treat all", because they avoid treating the
very few patients with predicted risk below 1%.

#### Caveat — transfer bias

These curves were computed on the primary outcome (`mortalidade_hospitalar`).
If cardiovascular deaths transferred out are coded as discharge, the models' net benefit is slightly
underestimated for that subgroup — the model is evaluated against an outcome that does not capture all
real deaths. The analysis with `mortalidade_hospitalar_ajustada` in Phase 5 will provide the most
accurate estimate of the true clinical benefit.

#### Methodological reference

Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating
prediction models. *Med Decis Making*. 2006;26(6):565–574.

## 12 · Save models and artifacts

In [ ]:
import joblib

for (nome_set, nome_motor, nome_validacao), info in modelos_gb.items():
    if nome_validacao != 'temporal':
        continue
    nome_arquivo = f"modelo_{nome_set.split()[1]}_{nome_motor}.joblib".replace('(', '').replace(')', '')
    joblib.dump(info['modelo'], f'{OUTDIR}/{nome_arquivo}')

with open(f'{OUTDIR}/modelo_A_fase4_melhores_motores.json', 'w', encoding='utf-8') as f:
    json.dump(melhor_por_set, f, ensure_ascii=False, indent=2)

print('Modelos, tabelas e figuras salvos em:', OUTDIR)
for arq in sorted(os.listdir(OUTDIR)):
    print(' -', arq)

---
### What this phase delivered
- Gradient boosting (XGBoost, LightGBM, CatBoost) tuned on both sets, with categoricals and missingness
  handled natively by each engine, without one-hot, without imputation, and without imbalance correction.
- Hyperparameters tuned by `StratifiedGroupKFold` (5 folds), on each validation's training split only.
- Direct comparison with the Phase 3 baseline (logistic regression) and with SAPS3, under both validations.
- DeLong test against SAPS3, for each set's best engine.
- Fine calibration (slope, intercept, Brier score decomposition) and decision curve.
- SHAP for both sets, separately.

### Next steps (Phase 5)
- Fairness — performance by subgroups (age, sex, admission type).
- If calibration reveals systematic bias, post-hoc recalibration (Platt scaling / isotonic), fitted only
  on the training/validation set.
- Reporting per TRIPOD-AI and risk-of-bias assessment with PROBAST.
- Sensitivity: re-including the 1,807 non-index admissions, with cross-validation grouped by patient.

### Limitation of this phase — transfer bias
All modeling (AUROC, DeLong, SHAP, calibration, decision curve) was run on the **primary definition** of
`mortalidade_hospitalar`. The parallel analysis with `mortalidade_hospitalar_ajustada` in Phase 5 will
allow quantifying the bias's impact on this phase's results. Record in PROBAST, outcome domain.

The SHAP importance check of the cardiovascular subgroup (§10) should be revisited after the adjusted
analysis: if `comorb_cardiovascular` changes importance between the two definitions, this confirms that
the bias affected what the model learned.